<a href="https://colab.research.google.com/github/treborskrub/Multi-agent-/blob/main/aiagenteval1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import math
import json
from typing import Dict, List, Any, Tuple

class StandardTopologicalEngine:
    def __init__(self, agent_count: int = 4):
        self.clock = 0
        self.agents = {i: {"coherence": 0.5, "arousal": 0.2, "sieve_active": False} for i in range(agent_count)}
        self.raw_accumulators = {i: {"coherence": 0.5, "arousal": 0.2} for i in range(agent_count)}
        self.memory_tensors = {i: {"coherence": 0.5} for i in range(agent_count)}
        self.adjacency_matrix = {i: {j: 1.0 for j in range(agent_count) if i != j} for i in range(agent_count)}

        self.cluster_adhesion_threshold = 0.28
        self.sieve_tolerance_threshold = 0.65
        self.decoherence_sieve_gain = 0.45
        self.state = {"global_entropy": 0.0}

    def _sigmoidal_clamp(self, val: float) -> float:
        return 1.0 / (1.0 + math.exp(-6.0 * (val - 0.5)))

    def step(self, signals: Dict[int, float], anomalies: Dict[int, float]) -> Dict[str, Any]:
        self.clock += 1
        prior_agents = {i: dict(a) for i, a in self.agents.items()}
        prior_memories = {i: dict(m) for i, m in self.memory_tensors.items()}

        # 1. Update Adjacency Matrix (Topological Edge Fission)
        for i in self.agents.keys():
            for j in self.agents.keys():
                if i == j: continue
                dist = abs(prior_memories[i]["coherence"] - prior_memories[j]["coherence"])
                self.adjacency_matrix[i][j] = 1.0 if dist <= self.cluster_adhesion_threshold else 0.0

        # 2. State Integration Loop
        for i, agent in self.agents.items():
            raw = self.raw_accumulators[i]
            memory = self.memory_tensors[i]

            # Arousal Kinematics
            raw["arousal"] += 0.4 * ((0.2 + anomalies[i]) - raw["arousal"])
            agent["arousal"] = self._sigmoidal_clamp(raw["arousal"])

            # Peer Graph Coupling Forces
            peer_pull, total_weight = 0.0, 0.0
            for j in self.agents.keys():
                if i == j or self.adjacency_matrix[i][j] == 0.0: continue
                dist = abs(memory["coherence"] - prior_memories[j]["coherence"])
                weight = math.exp(-2.0 * dist)
                peer_pull += weight * (prior_agents[j]["coherence"] - prior_agents[i]["coherence"])
                total_weight += weight

            normalized_drift = (peer_pull / total_weight) if total_weight > 0 else 0.0

            # Coherence Accumulation
            raw["coherence"] += (0.35 * (signals[i] - raw["coherence"])) + (0.40 * normalized_drift)

            # Sieve Pass
            tentative_coherence = self._sigmoidal_clamp(raw["coherence"])
            hallucination_index = tentative_coherence * (1.0 - signals[i])

            if hallucination_index >= self.sieve_tolerance_threshold:
                raw["coherence"] -= (self.decoherence_sieve_gain * hallucination_index)
                agent["coherence"] = self._sigmoidal_clamp(raw["coherence"])
                agent["sieve_active"] = True
            else:
                agent["coherence"] = tentative_coherence
                agent["sieve_active"] = False

            memory["coherence"] = (0.70 * memory["coherence"]) + (0.30 * agent["coherence"])

        # 3. Meta-Agent Macro Supervision
        coherences = [a["coherence"] for a in self.agents.values()]
        mean_coh = sum(coherences) / len(coherences)
        variance = sum((c - mean_coh)**2 for c in coherences) / len(coherences)
        self.state["global_entropy"] = variance

        if variance > 0.03:
            self.cluster_adhesion_threshold = max(0.12, self.cluster_adhesion_threshold - 0.015)
            self.decoherence_sieve_gain = min(0.85, self.decoherence_sieve_gain + 0.04)
        else:
            self.cluster_adhesion_threshold = min(0.28, self.cluster_adhesion_threshold + 0.008)
            self.decoherence_sieve_gain = max(0.45, self.decoherence_sieve_gain - 0.008)

        return {
            "tick": self.clock,
            "entropy": variance,
            "adhesion_limit": self.cluster_adhesion_threshold,
            "sieve_gain": self.decoherence_sieve_gain
        }

if __name__ == "__main__":
    engine = StandardTopologicalEngine()

    # Injected Evaluation Stream: Agent 2 suffers an ungrounded breakdown at Tick 3
    evaluation_stream = [
        {"signals": {0:0.8, 1:0.8, 2:0.8, 3:0.8}, "anomalies": {0:0, 1:0, 2:0, 3:0}}, # Frame 1: Nominal
        {"signals": {0:0.8, 1:0.8, 2:0.8, 3:0.8}, "anomalies": {0:0, 1:0, 2:0, 3:0}}, # Frame 2: Nominal
        {"signals": {0:0.8, 1:0.8, 2:0.1, 3:0.8}, "anomalies": {0:0, 1:0, 2:4.0, 3:0}} # Frame 3: Chaos on Agent 2
    ]

    print("======================================================================")
    print("        EXECUTING STANDARDIZED TOPOLOGICAL EVALUATION RUN             ")
    print("======================================================================")

    for frame in evaluation_stream:
        metrics = engine.step(frame["signals"], frame["anomalies"])
        print(f"\n[CLOCK TICK {metrics['tick']}] Global Entropy Field: {metrics['entropy']:.5f}")
        print(f"  -> Active Sieve Alerts: {[i for i, a in engine.agents.items() if a['sieve_active']]}")
        print(f"  -> Cluster Adhesion Limit: {metrics['adhesion_limit']:.4f}")
        print(f"  -> Node 2 Coherence Vector: {engine.agents[2]['coherence']:.4f}")

        EXECUTING STANDARDIZED TOPOLOGICAL EVALUATION RUN             

[CLOCK TICK 1] Global Entropy Field: 0.00000
  -> Active Sieve Alerts: []
  -> Cluster Adhesion Limit: 0.2800
  -> Node 2 Coherence Vector: 0.6525

[CLOCK TICK 2] Global Entropy Field: 0.00000
  -> Active Sieve Alerts: []
  -> Cluster Adhesion Limit: 0.2800
  -> Node 2 Coherence Vector: 0.7388

[CLOCK TICK 3] Global Entropy Field: 0.02014
  -> Active Sieve Alerts: []
  -> Cluster Adhesion Limit: 0.2800
  -> Node 2 Coherence Vector: 0.4590
